In [35]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pandas as pd

load_dotenv(override=True)
engine = create_engine(
    f"postgresql+psycopg2://{os.environ['DB_USER']}:{os.environ['DB_PASSWORD']}"
    f"@{os.environ['DB_HOST']}:{os.environ['DB_PORT']}/{os.environ['DB_NAME']}",
    pool_pre_ping=True,     # ping the pooled conn before use; silently replace dead ones
    pool_recycle=300,       # and recycle anything older than 5 min
)
pd.read_sql("SELECT 1 AS ok", engine)

python-dotenv could not parse statement starting at line 17


,ok
0,1


Setup financials & tickers for hiring dataset

In [36]:
fin = pd.read_sql("SELECT * FROM financials_quarterly ORDER BY ticker, quarter", engine)
fin["quarter"] = pd.to_datetime(fin["quarter"])
fin["rd_intensity"] = fin["rd_spend"] / fin["revenue"]

HIRING_TICKERS = ["AMD","AVGO","CDNS","INTC","MRVL","MU","NVDA","QCOM","TXN"]
core = fin[fin["ticker"].isin(HIRING_TICKERS)].copy()
print(core.groupby("ticker").size().to_string())   # quarters per ticker

ticker
AMD     19
AVGO    20
CDNS    19
INTC    19
MRVL    20
MU      19
NVDA    20
QCOM    18
TXN     19


Hiring Snapshot completeness verificiation


In [37]:
df = pd.read_sql("""
    SELECT ticker, title
    FROM hiring_signals
    WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM hiring_signals)
""", engine)
print(df.shape, "|", df["ticker"].nunique(), "tickers")

(10893, 2) | 9 tickers


Classifier from 03_hiring_category.ipynb

    - Note : AI-flagged only means AI is within the title of the job, does not directly mean if the job is an AI job, or magnitude of relation/usage to and of AI

In [38]:
# Role/AI classifier — copied verbatim from 03_hiring_category so 20 stands alone.
# If these rules ever change, change them in both places (or factor into a module then).
import re
AI_PAT = re.compile(r"\b(ai|ml|machine learning|deep learning|neural|llm|large language model|generative|genai|computer vision|nlp|natural language|transformer|reinforcement learning)\b")

ROLE_RULES = [
    ("Product Management",        r"\b(product manager|product management|product owner|head of product|director of product)\b"),
    ("Program / Project Mgmt",    r"\b(program manager|project manager|technical program|tpm|scrum master|release manager|agile)\b"),
    ("Sales / Field / Marketing", r"\b(application engineer|applications engineer|field application|solution architect|solutions architect|solutions engineer|sales|account manager|account executive|account director|business development|customer success|field engineer|presales|marketing|developer relations|client manager|enablement)\b"),
    ("Data & Analytics",          r"\b(data scientist|data engineer|data analyst|analytics|business intelligence)\b"),
    ("Corporate / Ops / G&A",     r"\b(accountant|accounting|finance|financial|controller|human resources|hr|employee relations|recruit|talent acquisition|legal|counsel|paralegal|facilities|supply chain|supply management|procurement|sourcing|supplier management|supplier manager|buyer|purchasing|payroll|logistics|operations|cost control|business analyst|program analyst|it support|help desk|administrative|executive assistant|corporate communications)\b"),
    ("Research / Science",        r"\b(research scientist|researcher|scientist|applied scientist|principal scientist|research engineer|post-doc|postdoc|post doc)\b"),
    ("Verification & Validation", r"\b(verification|validation|formal verification|emulation)\b"),
    ("Software & Firmware",       r"\b(software|firmware|embedded|device driver|sdk|kernel|compiler|full stack|fullstack|developer|devops|site reliability)\b"),
    ("Manufacturing & Process",   r"\b(process|equipment|manufacturing|fab|hvm|yield|packaging|assembly|technician|reliability|failure analysis|test engineer|production|industrial|quality|supplier quality|mechanical|mfg|product engineer|product development|module|npi|amhs)\b"),
    ("Design (silicon/IC)",       r"\b(design|designer|analog|mixed signal|mixed-signal|rtl|layout|asic|soc|silicon|chip|dft|vlsi|circuit|standard cell|cad)\b"),
    ("Systems & Architecture",    r"\b(systems|system engineer|architect|architecture|integration|signal integrity|performance|hardware|board)\b"),
    ("Engineering (unspecified)", r"\b(engineer|engineering)\b"),
    ("Management (general)",      r"\b(manager|mgr|director|management|head of|vice president|vp|chief|supervisor)\b"),
]
ROLE_RULES = [(name, re.compile(pat)) for name, pat in ROLE_RULES]

def classify_role(title):
    t = title.lower()
    for name, pat in ROLE_RULES:
        if pat.search(t):
            return name
    return "Other / Unclassified"

df["role_bucket"] = df["title"].fillna("").apply(classify_role)
df["is_ai"]       = df["title"].fillna("").apply(lambda t: bool(AI_PAT.search(t.lower())))

print(f"Unclassified: {(df['role_bucket']=='Other / Unclassified').mean():.1%}   |   "
      f"AI-flagged: {df['is_ai'].mean():.1%} ({int(df['is_ai'].sum())} jobs)")

Unclassified: 4.0%   |   AI-flagged: 9.4% (1029 jobs)


Aggregates

In [39]:
ai = (df.groupby("ticker")["is_ai"]
        .agg(ai_jobs="sum", n_jobs="count"))
ai["ai_pct"] = (ai["ai_jobs"] / ai["n_jobs"] * 100).round(1)
ai = ai.sort_values("ai_pct", ascending=False)

mix = pd.crosstab(df["ticker"], df["role_bucket"], normalize="index").mul(100).round(1)
mix = mix[df["role_bucket"].value_counts().index]

print("AI share per ticker:")
print(ai.to_string())
print("\nRole mix (% of each ticker's postings) — buckets as rows:")
print(mix.T.to_string())

AI share per ticker:
        ai_jobs  n_jobs  ai_pct
ticker                         
NVDA        465    2547    18.3
QCOM        268    1751    15.3
AMD         124    1048    11.8
INTC         44     648     6.8
CDNS         30     595     5.0
AVGO         10     335     3.0
MU           72    2846     2.5
TXN           8     517     1.5
MRVL          8     606     1.3

Role mix (% of each ticker's postings) — buckets as rows:
ticker                      AMD  AVGO  CDNS  INTC  MRVL    MU  NVDA  QCOM   TXN
role_bucket                                                                    
Manufacturing & Process     8.8  19.1   9.6  22.5   7.8  34.5   7.6   6.1  31.1
Design (silicon/IC)        20.0  25.7  19.2  20.7  31.7  10.4  11.9  17.1  16.4
Software & Firmware        16.1  14.0  22.5   8.3   6.1   2.5  26.3  20.3   4.6
Engineering (unspecified)   5.7  10.4   7.2   8.3   7.8  19.5   8.8  19.1   7.4
Verification & Validation  14.7   5.7  11.1  11.3  22.4   4.1   9.5   7.2   7.4
Sales / 

In [40]:
# ── Financial side: collapse each ticker to ONE trailing-4-quarter (TTM) row ──
# operating_margin is stored as a fraction; rebuild quarterly operating income so
# the TTM ratios are revenue-weighted (Σ ÷ Σ), not a mean of quarterly ratios.
core = core.sort_values(["ticker", "quarter"])
last4 = core.groupby("ticker").tail(4).copy()

# completeness guard — want q=4 and zero NaNs per ticker before trusting the sums
chk = last4.groupby("ticker").agg(
    q=("quarter", "size"),
    rev_na=("revenue", lambda s: s.isna().sum()),
    rd_na=("rd_spend", lambda s: s.isna().sum()),
    om_na=("operating_margin", lambda s: s.isna().sum()),
)
print("TTM window (want q=4, all *_na=0):")
print(chk.to_string())
print(f"\nwindow spans {last4['quarter'].min().date()} … {last4['quarter'].max().date()} "
      f"(fiscal calendars differ, so per-ticker windows vary slightly)\n")

last4["op_income"] = last4["operating_margin"] * last4["revenue"]
g = last4.groupby("ticker")
# match revenue to R&D availability → numerator and denominator span the SAME quarters
rev_rd = last4.loc[last4["rd_spend"].notna()].groupby("ticker")["revenue"].sum()
fin_ttm = pd.DataFrame({
    "rd_intensity":     g["rd_spend"].sum() / rev_rd,                    # both over same quarters
    "rd_qtrs":          g["rd_spend"].apply(lambda s: s.notna().sum()),  # quarters backing it (4 = full)
    "operating_margin": g["op_income"].sum() / g["revenue"].sum(),       # fraction; ×100 for %
})

# ── Merge: hiring features + TTM financial profile, one row per ticker ──
SEGMENT = {"NVDA":"Fabless","AMD":"Fabless","QCOM":"Fabless","AVGO":"Fabless","MRVL":"Fabless",
           "INTC":"IDM","MU":"IDM","TXN":"IDM","CDNS":"EDA"}

signals = (ai[["n_jobs", "ai_pct"]]
             .join(mix)          # role-mix % columns
             .join(fin_ttm))     # rd_intensity + operating_margin (TTM, fractions)
signals.insert(0, "segment", signals.index.map(SEGMENT))

print(signals.shape, "→ 9 tickers ×", signals.shape[1], "cols")
signals.round(3)

TTM window (want q=4, all *_na=0):
        q  rev_na  rd_na  om_na
ticker                         
AMD     4       0      0      0
AVGO    4       0      0      0
CDNS    4       0      0      0
INTC    4       0      0      0
MRVL    4       0      0      0
MU      4       0      0      0
NVDA    4       0      0      0
QCOM    4       0      0      0
TXN     4       0      0      0

window spans 2025-06-28 … 2026-05-28 (fiscal calendars differ, so per-ticker windows vary slightly)

(9, 20) → 9 tickers × 20 cols


,segment,n_jobs,ai_pct,Manufacturing & Process,Design (silicon/IC),Software & Firmware,Engineering (unspecified),Verification & Validation,Sales / Field / Marketing,Systems & Architecture,Corporate / Ops / G&A,Other / Unclassified,Program / Project Mgmt,Management (general),Product Management,Research / Science,Data & Analytics,rd_intensity,rd_qtrs,operating_margin
ticker,,,,,,,,,,,,,,,,,,,,
NVDA,Fabless,2547,18.3,7.6,11.9,26.3,8.8,9.5,12.8,8.6,2.2,1.8,3.6,1.9,1.6,3.0,0.4,0.082,4,0.640
QCOM,Fabless,1751,15.3,6.1,17.1,20.3,19.1,7.2,4.6,10.1,3.8,4.5,1.9,1.2,2.0,1.2,0.9,0.214,4,0.255
AMD,Fabless,1048,11.8,8.8,20.0,16.1,5.7,14.7,7.1,9.8,4.7,2.9,5.4,2.0,1.2,1.0,0.6,0.234,4,0.117
INTC,IDM,648,6.8,22.5,20.7,8.3,8.3,11.3,6.9,4.8,7.9,4.0,2.2,1.7,0.5,0.5,0.5,0.251,4,-0.094
CDNS,EDA,595,5.0,9.6,19.2,22.5,7.2,11.1,16.6,5.2,2.9,3.9,1.0,0.5,0.3,0.0,0.0,0.332,4,0.283
AVGO,Fabless,335,3.0,19.1,25.7,14.0,10.4,5.7,7.2,6.0,2.7,3.3,1.5,2.4,1.8,0.0,0.3,0.159,4,0.434
MU,IDM,2846,2.5,34.5,10.4,2.5,19.5,4.1,2.4,2.9,10.2,6.1,1.7,2.7,0.6,0.0,2.4,0.053,4,0.656
TXN,IDM,517,1.5,31.1,16.4,4.6,7.4,7.4,13.7,4.1,7.0,6.4,0.0,1.4,0.0,0.0,0.6,0.138,4,0.265
MRVL,Fabless,606,1.3,7.8,31.7,6.1,7.8,22.4,5.4,5.3,3.8,3.1,3.0,3.0,0.5,0.2,0.0,0.255,4,0.160


In [41]:
from sqlalchemy import text
q4_rd = 1_768_772_000 - 1_304_190_000   # FY2025 annual − 9mo YTD = 464,582,000
with engine.begin() as conn:
    n = conn.execute(text("""
        UPDATE financials_quarterly SET rd_spend = :v
        WHERE ticker='CDNS' AND quarter='2025-12-31' AND rd_spend IS NULL
    """), {"v": q4_rd}).rowcount
print(f"updated {n} row(s) — CDNS Q4 2025 R&D = {q4_rd/1e9:.3f}B")

updated 0 row(s) — CDNS Q4 2025 R&D = 0.465B


Q4 was recorded as NULL for Cadence

    - Q4 R&D was tagged under a different XBRL concept in EGDAR, ran through Claude to debug and made a quick one-row idempotent Update to patch.

    - Must fix backfill to scan all R&D concepts for the annual not just the first.

In [42]:
hire = ["ai_pct", "Software & Firmware"]
fin_cols = ["rd_intensity", "operating_margin"]

# Spearman (rank) is the primary read here: n=9 with real extremes (NVDA, MU),
# and the question is "do they move together in rank," not "linearly."
sp = signals[hire + fin_cols].corr(method="spearman").loc[hire, fin_cols]
pe = signals[hire + fin_cols].corr(method="pearson").loc[hire, fin_cols]

print("Spearman (rank):");  print(sp.round(2).to_string())
print("\nPearson (linear):"); print(pe.round(2).to_string())
print(f"\nai_pct ~ software/firmware: "
      f"spearman {signals['ai_pct'].corr(signals['Software & Firmware'], method='spearman'):.2f} | "
      f"pearson {signals['ai_pct'].corr(signals['Software & Firmware'], method='pearson'):.2f}")
print("n = 9 — directional, not significance-grade.")

Spearman (rank):
                     rd_intensity  operating_margin
ai_pct                      -0.08             -0.05
Software & Firmware          0.25              0.03

Pearson (linear):
                     rd_intensity  operating_margin
ai_pct                      -0.10              0.13
Software & Firmware          0.22              0.18

ai_pct ~ software/firmware: spearman 0.80 | pearson 0.78
n = 9 — directional, not significance-grade.


Analysis

    - Looking into AI-PCT (percentages of "AI" in job titles) and comparing it to % of Software/Firmware roles and operating margin & R&D Intensity (R&D Spend/Revenue). I found that there is no concrete correlation between operating margin and the other 2 but that there was a correlation between AI-PCT and % of Software/Firmware roles.
        - Remember that Correlation does NOT = Causation but this could hint at more software/firmware dominated companies being more invested within utilizing AI and hiring jobs with AI integration to be able to improve their software/firmware products.
    - Conducted a Spearman rank-correlation analysis to verify the eye-test I performed and my eye-test held true as there was a a relatively strong correlation between AI PCT and software/firmware % of jobs through both spearman and pearson at 0.8 and 0.78 respectively. In regards to the financials and their correlations to AI PCT and Software/Firmware % of jobs, there is no strong correlations given the numbers being close to 0.

Spearman = Correlation pertaining to how close two variabes move together in the same order
Pearson = Correlation pertaining to how much two variables move together in a straight line


In [56]:
import sys, os
sys.path.insert(0, os.path.abspath("../etl"))   # -> c:\Users\Johan\sector-signals\etl

import importlib, edgar_backfill
importlib.reload(edgar_backfill)

facts = edgar_backfill.fetch_company_facts("0000813672")   # CDNS
rd = edgar_backfill.extract_quarterly_series(
    facts, edgar_backfill.CONCEPT_CANDIDATES["rd_spend"]
)
from datetime import date
print("Q4 2025 present:", date(2025, 12, 31) in rd)
print("value:", rd.get(date(2025, 12, 31)))   # expect ~464,582,000

SyntaxError: closing parenthesis ']' does not match opening parenthesis '{' on line 53 (edgar_backfill.py, line 65)